In [ ]:
# Create a Spark session with local execution using all available cores
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("joins and Repartitioning")
    .master("local[*]")
    .getOrCreate()
)

In [ ]:
# Employee dataset
emp_data_1=[
     ["001","101","vaibhav","21","male","1200000"],
    ["002","102","dipak","22","male","20000"],
    ["003","103","tejas","23","female","2095"],
    ["004","104","anita","24","female","35000"],
    ["005","105","rohit","25","male","45000"],
    ["006","106","sneha","26","female","50000"],
    ["007","107","rahul","27","male","60000"],
    ["008","108","priya","28","female","70000"],
    ["009","109","amit","29","male","80000"],
    ["010","110","kavita","30","female","90000"],
    ["011","111","vikas","31","male","100000"],
    ["012","112","neha","32","female","110000"]
]
emp_schema_1="employee_id string,dept_id string,name string,age string,gender string,salary string"

emp_data_2=[
    ["1","vaibhav","101","shirdi"],
    ["2","dipak","102","mumbai"],
    ["3","aryan","103","panvel"],
    ["4","tejas","101","magarpatta"],
    ["5","roshan","102","korhale"]
]
emp_schema_2="id string,name string,dept_id string,city string"

In [ ]:
# Creating DataFrame for Above data
emp=spark.createDataFrame(data=emp_data_1,schema=emp_schema_1)
emp=spark.createDataFrame(data=emp_data_2,schema=emp_schema_1)

In [4]:
emp.show()

+---+-------+-------+----------+
| id|   name|dept_id|      city|
+---+-------+-------+----------+
|  1|vaibhav|    101|    shirdi|
|  2|  dipak|    102|    mumbai|
|  3|  aryan|    103|    panvel|
|  4|  tejas|    101|magarpatta|
|  5| roshan|    102|   korhale|
+---+-------+-------+----------+



In [ ]:
# Get current number of partitions in the DataFrame
emp.rdd.getNumPartitions()

8

In [ ]:
# Reaprtition the given DataFrame
emp_parttioned=emp.repartition(6)

In [12]:
emp_parttioned.rdd.getNumPartitions()

100

In [ ]:
# Reducing number of partition using Coalesce function
emp_par=emp.coalesce(2)

In [20]:
emp_par.rdd.getNumPartitions()

2

In [ ]:
# Import Spark Partition and add column showing partition
from pyspark.sql.functions import spark_partition_id

emp_1 = emp.withColumn("partition_num",spark_partition_id())

In [31]:
emp_1.show()

+---+-------+-------+----------+-------------+
| id|   name|dept_id|      city|partition_num|
+---+-------+-------+----------+-------------+
|  1|vaibhav|    101|    shirdi|            1|
|  2|  dipak|    102|    mumbai|            3|
|  3|  aryan|    103|    panvel|            4|
|  4|  tejas|    101|magarpatta|            6|
|  5| roshan|    102|   korhale|            7|
+---+-------+-------+----------+-------------+



In [34]:
emp_2=emp.repartition(4,"dept_id").withColumn("partition_num",spark_partition_id())

In [35]:
emp_2.show()

+---+-------+-------+----------+-------------+
| id|   name|dept_id|      city|partition_num|
+---+-------+-------+----------+-------------+
|  2|  dipak|    102|    mumbai|            0|
|  5| roshan|    102|   korhale|            0|
|  1|vaibhav|    101|    shirdi|            3|
|  3|  aryan|    103|    panvel|            3|
|  4|  tejas|    101|magarpatta|            3|
+---+-------+-------+----------+-------------+

